In [1]:
# ============================================================
# ATLAS DR2 — REu1 kNN REDSHIFT REGRESSION
# Reproduction of Luken et al. (2022)
# Based on the paper + official GitHub implementation
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

from astropy.io import fits

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error


# ============================================================
# 2. READ ATLAS FITS FILE
# ============================================================

file_path = "/home/sk/Projects/Astrophysics/Redshift-kNN-2021/ATLAS_complete_DR2.fits"

with fits.open(file_path) as hdul:

    # HDU 1 contains the ATLAS catalog
    data = hdul[1].data

# Convert FITS table to Pandas DataFrame
df = pd.DataFrame(data)

print("Original dataset shape:", df.shape)


# ============================================================
# 3. FEATURES USED IN THE PAPER / GITHUB REPOSITORY
# ============================================================

features = [
    "Sp2",
    "flux_ap2_36",
    "flux_ap2_45",
    "flux_ap2_58",
    "flux_ap2_80",
    "MAG_APER_4_G",
    "MAG_APER_4_R",
    "MAG_APER_4_I",
    "MAG_APER_4_Z"
]

target = "z"


# Check that all required columns exist
missing_columns = [
    col for col in features + [target]
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns: {missing_columns}"
    )


print("\nFeatures used:")
for i, feature in enumerate(features, 1):
    print(f"{i}. {feature}")

print(f"\nTarget: {target}")


# ============================================================
# 4. CREATE X AND y
# ============================================================

X = df[features].to_numpy(dtype=np.float64)
y = df[target].to_numpy(dtype=np.float64)

print("\nX shape:", X.shape)
print("y shape:", y.shape)


# ============================================================
# 5. CHECK FOR INVALID / MISSING VALUES
# ============================================================

if np.isnan(X).any():
    raise ValueError("NaN values found in X.")

if np.isnan(y).any():
    raise ValueError("NaN values found in y.")

if np.isinf(X).any():
    raise ValueError("Infinite values found in X.")

if np.isinf(y).any():
    raise ValueError("Infinite values found in y.")


# ============================================================
# 6. EXACT 70/30 DATA PARTITION
#    random_state = 42
# ============================================================

np.random.seed(42)

test_indices = np.random.choice(
    len(X),
    round(len(X) * 0.30),
    replace=False
)

train_indices = np.array(
    list(
        set(range(len(X))) -
        set(test_indices)
    )
)

X_train = X[train_indices]
X_test = X[test_indices]

y_train = y[train_indices]
y_test = y[test_indices]


print("\n==============================")
print("DATA PARTITION")
print("==============================")

print("Total sources   :", len(X))
print("Training sources:", len(X_train))
print("Testing sources :", len(X_test))


# ============================================================
# 7. 10-FOLD CROSS-VALIDATION
#    random_state = 10
# ============================================================

nSplits = 10

kFold = KFold(
    n_splits=nSplits,
    random_state=10,
    shuffle=True
)


# ============================================================
# 8. k VALUES USED BY THE PAPER
# ============================================================

neighboursList = list(
    range(2, 20)
)

print("\nTesting k values:")
print(neighboursList)


# ============================================================
# 9. CROSS-VALIDATION
#
#    For every k:
#      - 10-fold CV
#      - normalize using CV training fold
#      - train Euclidean kNN
#      - calculate eta_0.15
#      - calculate R²
#
#    Best k = minimum mean eta_0.15
# ============================================================

FailureLimit = 0.15

mse_final = []
outlier_final = []


for numNeighbours in neighboursList:

    mseList = []
    failed = []


    # --------------------------------------------------------
    # 10-fold CV
    # --------------------------------------------------------

    for trainIndex, testIndex in kFold.split(X_train):

        # CV training data
        X_train_cross = X_train[trainIndex].copy()
        y_train_cross = y_train[trainIndex]

        # CV validation data
        X_test_cross = X_train[testIndex].copy()
        y_test_cross = y_train[testIndex]


        # ----------------------------------------------------
        # NORMALIZATION
        #
        # Mean and standard deviation are calculated from
        # the CV training fold only.
        # ----------------------------------------------------

        for i in range(X_train_cross.shape[1]):

            mean = np.mean(
                X_train_cross[:, i]
            )

            std = np.std(
                X_train_cross[:, i]
            )

            X_train_cross[:, i] = (
                X_train_cross[:, i] - mean
            ) / std

            X_test_cross[:, i] = (
                X_test_cross[:, i] - mean
            ) / std


        # ----------------------------------------------------
        # EUCLIDEAN kNN REGRESSOR
        #
        # p = 2 → Euclidean distance
        # ----------------------------------------------------

        neigh = KNeighborsRegressor(
            n_neighbors=numNeighbours,
            p=2
        )

        neigh.fit(
            X_train_cross,
            y_train_cross
        )


        # ----------------------------------------------------
        # PREDICTION
        # ----------------------------------------------------

        pred = neigh.predict(
            X_test_cross
        )


        # ----------------------------------------------------
        # eta_0.15
        #
        # |z_pred - z_spec|
        # >
        # 0.15 * (1 + z_spec)
        # ----------------------------------------------------

        error = np.abs(
            pred - y_test_cross
        )

        catastrophic = np.sum(
            error >
            FailureLimit *
            (1 + y_test_cross)
        )

        failure_rate = (
            catastrophic /
            len(pred)
        )

        failed.append(
            failure_rate
        )


        # ----------------------------------------------------
        # R²
        # ----------------------------------------------------

        r2 = neigh.score(
            X_test_cross,
            y_test_cross
        )

        mseList.append(
            np.round(r2, 3)
        )


    # --------------------------------------------------------
    # Average over the 10 folds
    # --------------------------------------------------------

    mean_r2 = np.mean(
        mseList
    )

    mean_outlier = np.mean(
        failed
    )

    mse_final.append(
        mean_r2
    )

    outlier_final.append(
        mean_outlier
    )


    print(
        f"k = {numNeighbours:2d} | "
        f"CV eta_0.15 = {mean_outlier:.5f} | "
        f"CV R² = {mean_r2:.5f}"
    )


# ============================================================
# 10. SELECT BEST k
#
#     The GitHub implementation chooses the k having the
#     minimum eta_0.15.
# ============================================================

bestKIndex = np.argmin(
    np.array(outlier_final)
)

bestK = neighboursList[
    bestKIndex
]

bestCVOutlier = outlier_final[
    bestKIndex
]


print("\n==============================")
print("BEST k")
print("==============================")

print("Best k:", bestK)

print(
    "Best CV eta_0.15:",
    bestCVOutlier
)


# ============================================================
# 11. FINAL NORMALIZATION
#
#     Calculate mean/std from COMPLETE TRAINING SET.
#     Apply the same values to the test set.
# ============================================================

X_train_norm = np.copy(
    X_train
)

X_test_norm = np.copy(
    X_test
)


for i in range(
    X_train.shape[1]
):

    mean = np.mean(
        X_train[:, i]
    )

    std = np.std(
        X_train[:, i]
    )


    X_train_norm[:, i] = (
        X_train[:, i] - mean
    ) / std


    X_test_norm[:, i] = (
        X_test[:, i] - mean
    ) / std


# ============================================================
# 12. FINAL kNN MODEL
# ============================================================

final_model = KNeighborsRegressor(
    n_neighbors=bestK,
    p=2
)


final_model.fit(
    X_train_norm,
    y_train
)


# ============================================================
# 13. FINAL TEST PREDICTIONS
# ============================================================

finalPrediction = final_model.predict(
    X_test_norm
)


# ============================================================
# 14. RESIDUALS
#
#     Paper:
#
#     delta_z =
#     (z_spec - z_photo) / (1 + z_spec)
# ============================================================

residuals = (
    y_test -
    finalPrediction
) / (
    1 + y_test
)


# ============================================================
# 15. eta_0.15
#
#     |z_photo - z_spec|
#     >
#     0.15 * (1 + z_spec)
# ============================================================

error = np.abs(
    finalPrediction -
    y_test
)


outlierRate = (
    100 *
    np.sum(
        error >
        0.15 *
        (1 + y_test)
    )
    /
    len(finalPrediction)
)


# ============================================================
# 16. eta_2sigma
# ============================================================

stdRes = np.std(
    residuals
)


outlierRateSigma = (
    100 *
    np.sum(
        np.abs(residuals) >
        2 * stdRes
    )
    /
    len(residuals)
)


# ============================================================
# 17. sigma
# ============================================================

sigma = np.std(
    residuals
)


# ============================================================
# 18. sigma_NMAD
#
#     MAD =
#     median(|x - median(x)|)
#
#     sigma_NMAD =
#     1.4826 * MAD
# ============================================================

median_residual = np.median(
    residuals
)


mad = np.median(
    np.abs(
        residuals -
        median_residual
    )
)


sigma_nmad = (
    1.4826 *
    mad
)


# ============================================================
# 19. R²
# ============================================================

r2 = final_model.score(
    X_test_norm,
    y_test
)


# ============================================================
# 20. MSE
# ============================================================

mse = mean_squared_error(
    y_test,
    finalPrediction
)


# ============================================================
# 21. FINAL RESULTS
# ============================================================

print("\n")
print("======================================")
print("       REu1 — FINAL RESULTS")
print("======================================")

print(
    f"Training sources : {len(y_train)}"
)

print(
    f"Testing sources  : {len(y_test)}"
)

print(
    f"Number of features: {X_train.shape[1]}"
)

print(
    f"Best k            : {bestK}"
)

print(
    f"eta_0.15          : {outlierRate:.2f}%"
)

print(
    f"eta_2sigma        : {outlierRateSigma:.2f}%"
)

print(
    f"R²                : {r2:.2f}"
)

print(
    f"MSE               : {mse:.2f}"
)

print(
    f"sigma             : {sigma:.2f}"
)

print(
    f"sigma_NMAD        : {sigma_nmad:.2f}"
)

print("======================================")

Original dataset shape: (1311, 122)

Features used:
1. Sp2
2. flux_ap2_36
3. flux_ap2_45
4. flux_ap2_58
5. flux_ap2_80
6. MAG_APER_4_G
7. MAG_APER_4_R
8. MAG_APER_4_I
9. MAG_APER_4_Z

Target: z

X shape: (1311, 9)
y shape: (1311,)

DATA PARTITION
Total sources   : 1311
Training sources: 918
Testing sources : 393

Testing k values:
[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
k =  2 | CV eta_0.15 = 0.09476 | CV R² = 0.58710
k =  3 | CV eta_0.15 = 0.09912 | CV R² = 0.59490
k =  4 | CV eta_0.15 = 0.09478 | CV R² = 0.58440
k =  5 | CV eta_0.15 = 0.08932 | CV R² = 0.59190
k =  6 | CV eta_0.15 = 0.09587 | CV R² = 0.58170
k =  7 | CV eta_0.15 = 0.09803 | CV R² = 0.57750
k =  8 | CV eta_0.15 = 0.09909 | CV R² = 0.58190
k =  9 | CV eta_0.15 = 0.10349 | CV R² = 0.57960
k = 10 | CV eta_0.15 = 0.10350 | CV R² = 0.56670
k = 11 | CV eta_0.15 = 0.10131 | CV R² = 0.55910
k = 12 | CV eta_0.15 = 0.10242 | CV R² = 0.54960
k = 13 | CV eta_0.15 = 0.10896 | CV R² = 0.54350
k = 14 | CV et

In [1]:
# ============================================================
# ATLAS DR2 — REu1 kNN REDSHIFT REGRESSION
# Simplified scikit-learn implementation
# ============================================================

import numpy as np
import pandas as pd

from astropy.io import fits

from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.model_selection import GridSearchCV

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor

from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    make_scorer
)


# ============================================================
# 1. SETTINGS
# ============================================================

FILE_PATH = (
    "/home/sk/Projects/Astrophysics/"
    "Redshift-kNN-2021/ATLAS_complete_DR2.fits"
)

RANDOM_STATE = 42
CV_RANDOM_STATE = 10
TEST_SIZE = 0.30
N_SPLITS = 10
FAILURE_LIMIT = 0.15


FEATURES = [
    "Sp2",
    "flux_ap2_36",
    "flux_ap2_45",
    "flux_ap2_58",
    "flux_ap2_80",
    "MAG_APER_4_G",
    "MAG_APER_4_R",
    "MAG_APER_4_I",
    "MAG_APER_4_Z"
]

TARGET = "z"


# ============================================================
# 2. READ THE FITS FILE
# ============================================================

with fits.open(FILE_PATH) as hdul:
    data = hdul[1].data

df = pd.DataFrame(data)

print("Original dataset shape:", df.shape)


# ============================================================
# 3. CHECK REQUIRED COLUMNS
# ============================================================

required_columns = FEATURES + [TARGET]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns: {missing_columns}"
    )


# ============================================================
# 4. CREATE FEATURES AND TARGET
# ============================================================

X = df[FEATURES].to_numpy(dtype=np.float64)
y = df[TARGET].to_numpy(dtype=np.float64)


# ============================================================
# 5. REMOVE INVALID ROWS
# ============================================================

valid_rows = (
    np.isfinite(X).all(axis=1)
    & np.isfinite(y)
)

X = X[valid_rows]
y = y[valid_rows]

print("Valid dataset shape:", X.shape)
print("Number of features:", X.shape[1])


# ============================================================
# 6. TRAIN/TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True
)

print("\nDATA PARTITION")
print("------------------------------")
print("Total sources   :", len(X))
print("Training sources:", len(X_train))
print("Testing sources :", len(X_test))


# ============================================================
# 7. DEFINE eta_0.15
# ============================================================

def eta_015(y_true, y_pred):
    """
    Fraction of objects satisfying:

    |z_pred - z_spec| > 0.15 * (1 + z_spec)
    """

    error = np.abs(y_pred - y_true)

    catastrophic = (
        error > FAILURE_LIMIT * (1 + y_true)
    )

    return np.mean(catastrophic)


def negative_eta_015(y_true, y_pred):
    """
    GridSearchCV maximizes scores.

    Therefore, return negative eta_0.15
    so that minimizing eta_0.15 becomes maximizing
    negative eta_0.15.
    """

    return -eta_015(y_true, y_pred)


eta_scorer = make_scorer(
    negative_eta_015,
    greater_is_better=True
)


# ============================================================
# 8. CREATE THE MODEL PIPELINE
# ============================================================

model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "knn",
            KNeighborsRegressor(
                weights="uniform",
                p=2
            )
        )
    ]
)


# ============================================================
# 9. DEFINE CROSS-VALIDATION
# ============================================================

cv = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=CV_RANDOM_STATE
)


# ============================================================
# 10. SEARCH FOR THE BEST k
# ============================================================

parameter_grid = {
    "knn__n_neighbors": list(range(2, 20))
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=parameter_grid,
    scoring=eta_scorer,
    cv=cv,
    refit=True,
    n_jobs=-1,
    return_train_score=False
)


grid_search.fit(X_train, y_train)


best_model = grid_search.best_estimator_
best_k = grid_search.best_params_["knn__n_neighbors"]

best_cv_eta = -grid_search.best_score_


print("\nCROSS-VALIDATION RESULTS")
print("------------------------------")
print("Best k:", best_k)
print("Best CV eta_0.15:", best_cv_eta)


# ============================================================
# 11. DISPLAY RESULTS FOR EVERY k
# ============================================================

cv_results = pd.DataFrame(
    grid_search.cv_results_
)

cv_results["mean_eta_015"] = (
    -cv_results["mean_test_score"]
)

cv_results["k"] = (
    cv_results["param_knn__n_neighbors"]
    .astype(int)
)

cv_results = cv_results.sort_values("k")

print("\nALL k VALUES")
print("------------------------------")

print(
    cv_results[
        ["k", "mean_eta_015", "std_test_score"]
    ].to_string(index=False)
)


# ============================================================
# 12. FINAL TEST PREDICTIONS
# ============================================================

y_pred = best_model.predict(X_test)


# ============================================================
# 13. RESIDUALS
# ============================================================

# Paper definition:
#
# delta_z = (z_spec - z_photo) / (1 + z_spec)

residuals = (
    y_test - y_pred
) / (
    1 + y_test
)


# ============================================================
# 14. eta_0.15 ON TEST SET
# ============================================================

test_eta_015 = eta_015(
    y_test,
    y_pred
)

test_eta_015_percent = (
    100 * test_eta_015
)


# ============================================================
# 15. eta_2sigma
# ============================================================

sigma = np.std(residuals)

test_eta_2sigma = np.mean(
    np.abs(residuals) > 2 * sigma
)

test_eta_2sigma_percent = (
    100 * test_eta_2sigma
)


# ============================================================
# 16. sigma_NMAD
# ============================================================

median_residual = np.median(residuals)

mad = np.median(
    np.abs(residuals - median_residual)
)

sigma_nmad = 1.4826 * mad


# ============================================================
# 17. OTHER REGRESSION METRICS
# ============================================================

test_r2 = r2_score(
    y_test,
    y_pred
)

test_mse = mean_squared_error(
    y_test,
    y_pred
)


test_rmse = np.sqrt(test_mse)


# ============================================================
# 18. FINAL RESULTS
# ============================================================

print("\nFINAL TEST RESULTS")
print("======================================")
print("Training sources :", len(y_train))
print("Testing sources  :", len(y_test))
print("Number of features:", len(FEATURES))
print("Best k            :", best_k)
print(f"eta_0.15          : {test_eta_015_percent:.2f}%")
print(f"eta_2sigma        : {test_eta_2sigma_percent:.2f}%")
print(f"R²                : {test_r2:.4f}")
print(f"MSE               : {test_mse:.6f}")
print(f"RMSE              : {test_rmse:.6f}")
print(f"sigma             : {sigma:.6f}")
print(f"sigma_NMAD        : {sigma_nmad:.6f}")
print("======================================")

Original dataset shape: (1311, 122)
Valid dataset shape: (1311, 9)
Number of features: 9

DATA PARTITION
------------------------------
Total sources   : 1311
Training sources: 917
Testing sources : 394

CROSS-VALIDATION RESULTS
------------------------------
Best k: 6
Best CV eta_0.15: 0.08718346870520784

ALL k VALUES
------------------------------
 k  mean_eta_015  std_test_score
 2      0.100299        0.026496
 3      0.092630        0.022272
 4      0.099164        0.024324
 5      0.092642        0.027639
 6      0.087183        0.023757
 7      0.092642        0.023368
 8      0.090456        0.020530
 9      0.094828        0.021151
10      0.099212        0.019069
11      0.100287        0.016526
12      0.101386        0.019990
13      0.104646        0.022789
14      0.107943        0.019675
15      0.103595        0.023478
16      0.104682        0.023426
17      0.104694        0.024964
18      0.109054        0.024887
19      0.113426        0.025927

FINAL TEST RESULTS
